In [7]:
import glob, re
import numpy as np
import soundfile as sf
import torch
from pathlib import Path

In [ ]:
CKPT = "./vits_colombian/output/train5/best_model.pth"

OUTPUTS = Path().resolve()
OUTPUTS.mkdir(parents=True, exist_ok=True)

NOISE_SCALE = 0.8
NOISE_SCALE_W = 0.6
LENGTH_SCALE = 1.0

FRASES = [
    "Hola parce que dia tan bueno hace hoy, me voy a tomar un tinto y después salgo a dar una vuelta por el parque",
    "Hola buenos dias mano, en que puedo ayudarte hoy?",
]

In [9]:
best_ckpt = CKPT
config_path_ft = None
for parent in [Path(best_ckpt).parent, Path(best_ckpt).parent.parent, Path(best_ckpt).parent.parent.parent]:
    if (parent / "config.json").exists():
        config_path_ft = str(parent / "config.json")
        break

print(f"Checkpoint: {best_ckpt}")
print(f"Config: {config_path_ft}")

Checkpoint: ./vits_colombian/output/train5/best_model.pth
Config: vits_colombian/output/train5/config.json


In [10]:
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.models.vits import Vits
from TTS.tts.layers.vits.networks import TextEncoder

inf_config = VitsConfig()
inf_config.load_json(config_path_ft)

inf_model = Vits.init_from_config(inf_config)
ma = inf_config.model_args
inf_model.text_encoder = TextEncoder(
    n_vocab=ma.num_chars,
    out_channels=ma.hidden_channels,
    hidden_channels=192,
    hidden_channels_ffn=768,
    num_heads=ma.num_heads_text_encoder,
    num_layers=6,
    kernel_size=3,
    dropout_p=0.1,
    language_emb_dim=4,
)

inf_model.load_checkpoint(inf_config, best_ckpt, eval=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
inf_model = inf_model.to(DEVICE).eval()
print(DEVICE)

 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > initialization of language-embedding layers.
cpu


In [11]:
def extraer_wav(outputs):
    if isinstance(outputs, dict):
        for key in ["wav", "waveform", "audio", "model_outputs", "wav_seg"]:
            if key in outputs:
                return outputs[key]
        for v in outputs.values():
            if torch.is_tensor(v) and v.ndim >= 1:
                return v
        raise KeyError(f"Sin tensor de audio en: {list(outputs.keys())}")
    elif isinstance(outputs, (list, tuple)):
        return outputs[0]
    elif torch.is_tensor(outputs):
        return outputs
    raise TypeError(f"Formato desconocido: {type(outputs)}")

In [12]:
m = re.search(r'(\d+)\.pth$', best_ckpt)
step = m.group(1) if m else "best"
sr_out = inf_config.audio.sample_rate

for i, texto in enumerate(FRASES):
    try:
        tokens = inf_model.tokenizer.text_to_ids(texto)
        x = torch.LongTensor(tokens).unsqueeze(0).to(DEVICE)
        x_lengths = torch.LongTensor([x.shape[1]]).to(DEVICE)
        lang_ids = torch.LongTensor([0]).to(DEVICE)

        with torch.no_grad():
            outputs = inf_model.inference(
                x,
                aux_input={
                    "x_lengths"  : x_lengths,
                    "speaker_ids": None,
                    "language_ids": lang_ids,
                    "noise_scale" : NOISE_SCALE,
                    "noise_scale_w": NOISE_SCALE_W,
                    "length_scale" : LENGTH_SCALE,
                },
            )

        wav = extraer_wav(outputs)
        audio_np = wav.squeeze().cpu().numpy() if torch.is_tensor(wav) else np.asarray(wav).squeeze()
        peak = np.abs(audio_np).max()
        if peak > 0.001:
            audio_np = audio_np / peak * 0.92

        out_path = OUTPUTS / f"resultado_{i+1}.wav"
        sf.write(str(out_path), audio_np, sr_out)
        print(f"frase {i+1} - {out_path.name}")
        print(f"{texto}\n")

    except Exception as e:
        import traceback
        print(f"ERROR frase {i+1}: {texto}")
        traceback.print_exc()

frase 1 - resultado_1.wav
Hola parce que dia tan bueno hace hoy, me voy a tomar un tinto y después salgo a dar una vuelta por el parque

frase 2 - resultado_2.wav
Hola buenos dias mano, en que puedo ayudarte hoy?

